In [1]:
import fitz  # PyMuPDF
import re
import json
from typing import List, Dict, Optional
from dataclasses import dataclass, asdict
from pathlib import Path
import pandas as pd
from dataclasses import asdict

import openai

In [2]:
class PDFReaderPipeline:
    def __init__(self, pdf_path: str, api_key: str):
        self.pdf_path = pdf_path
        self.api_key = api_key
        self.pages_data: List
        self.gt_list = [
            "(주)스텝투미", "기장 아난티 루프탑 요가", "라자요가명상센터 해운대", "아누 요가", 
            "에스엠비 웰니스", "클럽디오아시스", "파크 하얏트 부산 루미 스파 피트니스", "히말라야 싱잉볼", 
            "아난티 맥퀸즈 클럽 고압산소 챔버", "리트릿 시그니엘 스파", "순다리 스파", "신에스테틱", 
            "약손명가", "뮤지엄 원", "부산근현대역사관", "부산박물관", "범어사 성보박물관", 
            "부산현대미술관", "국제갤러리", "도모헌", "소울아트스페이스", "아트소향", "이웰갤러리", 
            "조현화랑", "카린 갤러리", "드림씨어터", "부산콘서트홀", "영화의 전당", "미오드모이", 
            "롯데백화점 부산본점", "롯데백화점 센텀지점", "롯데백화점 광복점", "신세계백화점 센텀시티점", 
            "민속공예 나전칠기", "보명필방", "부산온도담", "설리화", "신미선한과", "율서", 
            "비아인키노", "에크루", "이용기 목조형연구소", "광안리 해양 레포츠 센터", "해운대리버크루즈", 
            "그리니어", "고유 디저트 스튜디오", "그레이스 쿠킹가든", "물빛색 스튜디오", "바이트 사이트", 
            "밝히는 사람들", "비비비당", "삼진어묵본점", "아이러브한식", "위닛", "율서", 
            "퍼스널컬러 라비앙수 부산", "향산재 문화원", "호댄스스튜디오", "금정산성 막걸리 체험", 
            "박명화 옻칠 스튜디오", "부산서예캘리그라피교육원", "수걸도예", "세븐럭카지노", "아르떼뮤지엄 부산"
        ]
        self.df: pd.DataFrame
        
    def read_pdf_with_pymupdf_split(self) -> List[Dict]:
        try:
            doc = fitz.open(self.pdf_path)
            pages_data = []
            
            only_right_pages = []
            only_left_pages = []

            for page_num in range(2, len(doc)):
                page = doc[page_num]
                rect = page.rect
                mid_x = rect.width / 2
                # 하단 페이지 번호 영역(약 50px)을 제외하기 위한 높이 설정


                # 좌/우 영역 정의 (하단 제외)
                if page_num in only_right_pages:
                    clips = [
                        fitz.Rect(mid_x, 0, rect.width, rect.height) # 오른쪽
                    ]
                
                elif page_num in only_left_pages:
                    clips = [
                        fitz.Rect(0, 0, mid_x, rect.height),      # 왼쪽
                    ]
                else:
                    clips = [
                        fitz.Rect(0, 0, mid_x, rect.height),      # 왼쪽
                        fitz.Rect(mid_x, 0, rect.width, rect.height) # 오른쪽
                    ]

                full_text = []
                full_text.append(f"\n--- PAGE {page_num + 1} START ---")
                for clip in clips:
                    # "dict" 형식으로 읽어야 글자 색상(color) 정보가 포함됨
                    blocks = page.get_text("blocks", clip=clip)
                    
                    for clip in clips:
                        # "blocks" 방식으로 변경: 문단 단위로 텍스트를 가져옵니다.
                        blocks = page.get_text("blocks", clip=clip)
                        
                        # blocks의 각 항목: (x0, y0, x1, y1, "텍스트 내용", block_no, block_type)
                        for b in blocks:
                            text_content = b[4].strip() # 4번째 인덱스가 텍스트입니다.
                            if text_content:
                                full_text.append(text_content)

                pages_data.append({
                    'page_num': page_num + 1,
                    'text': "\n".join(full_text).strip()
                })
            
            doc.close()
            print(f"✅ PDF 읽기 완료: 총 {len(pages_data)} 페이지")
            return pages_data
        
        except Exception as e:
            print(f"❌ 오류 발생: {e}")
            return []
        
    def read_pdf_with_pymupdf(self) -> List[Dict]:
        try:
            doc = fitz.open(self.pdf_path)
            pages_data = []

            for page_num in range(len(doc)):
                page = doc[page_num]
                
                # 1. 페이지 전체 텍스트 추출 (통으로 읽기)
                # "text" 모드는 페이지의 텍스트를 위에서 아래로 흐름대로 가져옵니다.
                text_content = page.get_text("text").strip()

                # 2. 결과 저장
                if text_content:
                    pages_data.append({
                        'page_num': page_num + 1,
                        'text': f"--- PAGE {page_num + 1} START ---\n{text_content}"
                    })
            
            doc.close()
            print(f"✅ PDF 읽기 완료: 총 {len(pages_data)} 페이지")
            return pages_data
        
        except Exception as e:
            print(f"❌ 오류 발생: {e}")
            return []
        
    def make_ground_truth(self, gt_list: List):
        self.gt_list = gt_list

        
    def refine_with_llm(self, text_to_process: str) -> str:
        client = openai.OpenAI(api_key=self.api_key)
        
        system_prompt = "당신은 텍스트 정제 전문가입니다. 설명이나 요약 없이 오직 지정된 포맷의 데이터만 출력하세요."
        
        user_prompt = f"""
[입력 텍스트]에서 장소 정보를 추출하여 [출력 포맷]에 맞게 정돈하세요.
내용을 절대 요약하거나 바꾸지 말고, 맞춤법만 교정하세요.
내용이 없다면 임의로 채우지 말고 비워 두세요.

# Guidelines
1. 영어 정보: 영어로 작성된 정보는 다 무시하세요.
2. 구분: 반드시 구분에는 관광지 / 음식점 / 카페 / 숙박업체 / 체험 / 문화예술 / 쇼핑 / 축제 중 하나로 엄격히 분류하세요. 장소명에 시장, 골목이 포함되면 관광지로 적으세요.
3. 카테고리: 음식점/카페인 경우 핵심 메뉴 타입(예: 돼지국밥 / 베이커리 / 커피전문점)을 적고, 관광지는 비워두세요. 숙박업체라면 숙박타입(예: 호텔 / 모텔)을 적으세요.
4. 메뉴: 음식점이 아니라면 프로그램, 가격정보를 적으세요
5. 원문 유지: 설명 글은 요약 없이 맞춤법과 띄어쓰기만 교정하여 그대로 유지하세요.
6. 무결성: 정보가 없는 항목은 임의로 채우지 말고 빈칸으로 두세요.
7. 반복 출력: 여러 장소가 포함된 경우 장소별로 구분하여 반복 작성하세요.
8. 지역 정보 추출: 특정 업장(식당/카페) 외에 '기장', '부리단길', '영도'와 같이 지역 전체를 설명하는 텍스트가 있다면, 이를 '장소명'으로 간주하여 반드시 별도 항목으로 추출하세요. 이 경우 '구분'은 '관광지'로 작성합니다.

[출력 포맷]
장소명: 
구분:
카테고리: 
설명: 
메뉴: 
영업시간: 
주소:
연락처:

[입력 텍스트]
{text_to_process}
"""

        try:
            response = client.chat.completions.create(
                model="gpt-4o",  # 또는 "gpt-3.5-turbo"
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0  # 일관성을 위해 0으로 설정
            )
            return response.choices[0].message.content
        except Exception as e:
            print(f"❌ GPT 정제 실패: {e}")
            return text_to_process # 실패 시 원본 반환
        
    def change_str_to_df(self, full_text: str) -> pd.DataFrame:   
        keys = ["장소명", "구분", "카테고리", "설명", "메뉴", "영업시간", "주소", "연락처"]
        pattern = r"장소명:.*?(?=장소명:|$)" # 식당 단위로 먼저 분할

        restaurants = re.findall(pattern, full_text.replace('\n', ' '), re.DOTALL)
        data_list = []

        for res in restaurants:
            row = {}
            for i in range(len(keys)):
                start_key = keys[i] + ":"
                if i < len(keys) - 1:
                    end_key = keys[i+1] + ":"
                    item_pattern = f"{start_key}(.*?){end_key}"
                else:
                    item_pattern = f"{start_key}(.*)"
                
                match = re.search(item_pattern, res)
                if match:
                    row[keys[i]] = match.group(1).strip()
            data_list.append(row)

        self.df = pd.DataFrame(data_list)

        return self.df
        
    def evaluate_df(self) -> None:
        # 검증
        df_places = set(self.df['장소명'].str.strip())
        gt_places = set(self.gt_list)

        missing_items = list(gt_places - df_places)
        extra_items = list(df_places - gt_places)

        if not missing_items:
                print("✅ 모든 장소명이 포함되어 있습니다!")
        else:
            print(f"❌ 누락된 장소 ({len(missing_items)}개):")
            for item in sorted(missing_items):
                print(f"  - {item}")
                
        if extra_items:
            print(f"\n⚠️ GT 외 추가된 장소 ({len(extra_items)}개 - 확인 필요):")
            for item in sorted(extra_items):
                print(f"  - {item}")

    

        
    def save_to_json(self, file_name: str) -> None:
        file_name = file_name

        self.df.to_json(
            file_name, 
            orient='records', 
            force_ascii=False,   # 한글이 유니코드로 변환되지 않고 그대로 저장됨
            indent=4             # 사람이 보기 좋게 들여쓰기 적용 (파일 용량은 약간 늘어남)
        )

        print(f"✅ JSON 저장 완료: {file_name}")


In [ ]:
pdf_path = "./busan_trip_pdfs/부산골목길관광가이드북.pdf"
api_key = ""    # OpenAI api key

pdf_reader = PDFReaderPipeline(pdf_path=pdf_path, api_key=api_key)

## 룰베이스 pdf 읽기

In [15]:
pages_list = pdf_reader.read_pdf_with_pymupdf()

✅ PDF 읽기 완료: 총 0 페이지


In [13]:
pages_list[0]

IndexError: list index out of range

In [30]:
pre_pages_list, mid_pages_list ,post_pages_list = pages_list[:len(pages_list)//3], pages_list[len(pages_list)//3: len(pages_list)//3 * 2], pages_list[len(pages_list)//3 * 2 :]

print(f"split unit : {len(pages_list)//3}, {len(pages_list)//3 * 2}")

split unit : 3, 6


In [31]:
pre_text = '\n'.join([pre_pages_list[i]['text'] for i in range(len(pre_pages_list))])

In [32]:
mid_text = '\n'.join([mid_pages_list[i]['text'] for i in range(len(mid_pages_list))])

In [33]:
post_text = '\n'.join([post_pages_list[i]['text'] for i in range(len(post_pages_list))])

## LLM을 활용한 정제

In [34]:
pre_text

"--- PAGE 1 START ---\n6\n생겨나면서 가장 인기있는 문화골목이 되었다. \n동시에 아직까지 남아있는 옛 공구상가의 \n모습은  전포동의 역사를 그대로 엿볼 수 있는데 \n이런 것들이 자연스레 어우러져 색다른 골목 \n분위기를 만들고 있다. 알록달록한 색깔의 외관, \n아기자기한 소품, 예쁜 조명, 화려한 장식 등 \n시선을 사로잡는 다양한 가게들이 속속 들어선 \n전포공구길만의 공간은 인증사진을 남기고 싶어 \n하는 여행자들에게 많은 사랑을 받고 있다.\n'깡깡'하며 기계 고치는 소리와 \n'드르르륵' 커피 내리는 소리가 \n과거와 현재를 이어주는 묘한\n연결고리로 다가온다.\n과거 전포공구길은 차량 재생창*, 공업사 등의  \n시설에서 나온 부품들이 거래되면서 공구 골목을 \n형성하였다. 전포동 공구 산업이 활발하던 시절 \n공업사 종사자는 바쁜 일상으로 인해 빵과 우유로 \n끼니를 해결한 추억이 있다. 최근에는 곳곳에 \n카페, 맛집, 편집샵들이 생겨나 이색적이고 \n독특한 분위기로 주목을 받으며 젊은 사람들이 \n모여들어 힙한 골목길로 자리잡고 있다.\n*재생창: 군수품의 수리와 폐품 재생을 주로 하는 공창\n트렌디한 카페와 공방이 하나둘씩 들어서면서 \n‘힙’한 거리로 바뀌고 있는 전포공구길은 \n국내외 관광객이 찾는 핫플레이스가 되었다. \n각자의 개성을 뽐내는 <카페와 베이커리>, \n아기자기한 물건들로 가득한 <소품샵>, \n발란사나 아더에러 같은 <핫한 브랜드 상점>, \n다양한 나라의 <음식점>, 팝업 스토어 등이 \n전포공구길의 탄생\n--- PAGE 2 START ---\n12\n사진과 공방\n전포공구길 핫플레이스\n경성72스튜디오\nfun play Shutter, 놀고 기록하고 사랑하라\n서전로37번길 14 지하층  IG. @kyungsung72studio\n1\n수에노센트스튜디오\n전문 조향사의 캔들공방\n서전로37번길 26 A동 159호  IG. @sueno_candle\n2\n태한사진관\n소중한 순간을 기록하는 가

In [35]:
refined_pre_text = pdf_reader.refine_with_llm(pre_text)

In [36]:
print(refined_pre_text)

장소명: 전포공구길
구분: 관광지
카테고리: 
설명: 생겨나면서 가장 인기 있는 문화 골목이 되었다. 동시에 아직까지 남아있는 옛 공구 상가의 모습은 전포동의 역사를 그대로 엿볼 수 있는데 이런 것들이 자연스레 어우러져 색다른 골목 분위기를 만들고 있다. 알록달록한 색깔의 외관, 아기자기한 소품, 예쁜 조명, 화려한 장식 등 시선을 사로잡는 다양한 가게들이 속속 들어선 전포공구길만의 공간은 인증사진을 남기고 싶어 하는 여행자들에게 많은 사랑을 받고 있다. '깡깡'하며 기계 고치는 소리와 '드르르륵' 커피 내리는 소리가 과거와 현재를 이어주는 묘한 연결고리로 다가온다. 과거 전포공구길은 차량 재생창, 공업사 등의 시설에서 나온 부품들이 거래되면서 공구 골목을 형성하였다. 전포동 공구 산업이 활발하던 시절 공업사 종사자는 바쁜 일상으로 인해 빵과 우유로 끼니를 해결한 추억이 있다. 최근에는 곳곳에 카페, 맛집, 편집샵들이 생겨나 이색적이고 독특한 분위기로 주목을 받으며 젊은 사람들이 모여들어 힙한 골목길로 자리잡고 있다. 트렌디한 카페와 공방이 하나둘씩 들어서면서 ‘힙’한 거리로 바뀌고 있는 전포공구길은 국내외 관광객이 찾는 핫플레이스가 되었다. 각자의 개성을 뽐내는 카페와 베이커리, 아기자기한 물건들로 가득한 소품샵, 발란사나 아더에러 같은 핫한 브랜드 상점, 다양한 나라의 음식점, 팝업 스토어 등이 전포공구길의 탄생
메뉴: 
영업시간: 
주소:
연락처:

장소명: 경성72스튜디오
구분: 체험
카테고리: 
설명: fun play Shutter, 놀고 기록하고 사랑하라
메뉴: 
영업시간: 
주소: 서전로37번길 14 지하층
연락처: 

장소명: 수에노센트스튜디오
구분: 체험
카테고리: 
설명: 전문 조향사의 캔들공방
메뉴: 
영업시간: 
주소: 서전로37번길 26 A동 159호
연락처: 

장소명: 태한사진관
구분: 체험
카테고리: 
설명: 소중한 순간을 기록하는 가족사진관
메뉴: 
영업시간: 
주소: 서전로47번길 27
연락처: 

장소명: 어피얼
구분: 체험


In [37]:
refined_mid_text = pdf_reader.refine_with_llm(mid_text)

In [38]:
print(refined_mid_text)

장소명: 달구나
구분: 카페
카테고리: 베이커리
설명: 달콤한 수제 다쿠아즈와 쿠키
메뉴: 
영업시간: 
주소: 서전로37번길 20 1층
연락처: 

장소명: 백금당
구분: 카페
카테고리: 베이커리
설명: 소복히 쌓인 크림케이크로 당충전
메뉴: 
영업시간: 
주소: 서전로37번길 20 A마동 2층
연락처: 

장소명: 연의양과
구분: 카페
카테고리: 베이커리
설명: 매일 새롭게 구워내는 까눌레와 다쿠아즈
메뉴: 
영업시간: 
주소: 서전로37번길 20
연락처: 

장소명: 이너프
구분: 카페
카테고리: 커피전문점
설명: 맛있는 커피와 든든한 브런치
메뉴: 
영업시간: 
주소: 서전로37번길 26
연락처: 

장소명: 굿모닝홍콩
구분: 음식점
카테고리: 홍콩 음식
설명: 누구나 부담없이 즐길 수 있는 홍콩 음식
메뉴: 
영업시간: 
주소: 서전로47번길 19 1층
연락처: 

장소명: 꽃잎휴게실
구분: 카페
카테고리: 
설명: 오랜기간 전포공구길을 지켜준 휴게실
메뉴: 
영업시간: 
주소: 서전로37번길 20
연락처: 

장소명: 동백아가씨1961
구분: 음식점
카테고리: 즉석떡볶이
설명: 중독되는 맛깔나는 즉석떡볶이
메뉴: 
영업시간: 
주소: 서전로37번길 18
연락처: 

장소명: 버거샵
구분: 음식점
카테고리: 햄버거
설명: 레트로한 분위기의 수제 햄버거
메뉴: 
영업시간: 
주소: 동천로108번길 11
연락처: 

장소명: J.Brews
구분: 카페
카테고리: 
설명: 야외테이블이 매력적인 명화카페
메뉴: 
영업시간: 
주소: 서전로37번길 20
연락처: 

장소명: 이터널선샤인
구분: 카페
카테고리: 커피전문점
설명: 부드러운 커피와 어울리는 소금빵
메뉴: 
영업시간: 
주소: 서전로47번길 27 2층
연락처: 

장소명: 향미공간
구분: 카페
카테고리: 커피전문점
설명: 공간을 커피의 향으로 채우다
메뉴: 
영업시간: 
주소: 동천로108번길 41 2층
연락처: 

장소명: 훌드
구분: 카페
카테고리: 
설명: 책과 음료과 공존하는 다정한 공간
메뉴

In [39]:
refined_post_text = pdf_reader.refine_with_llm(post_text)

In [40]:
print(refined_post_text)

장소명: 서면먹자골목
구분: 관광지
카테고리: 
설명: 부산에서 유동인구가 가장 많은 서면 로터리와 태화쇼핑 건너편 뒷골목에 위치하는 곳으로 떡볶이, 오뎅, 순대를 비롯하여 계란, 파전, 만두, 김밥 등 다양한 길거리 음식과 분위기 좋은 카페, 호프집 등이 줄지어 맛있는 냄새로 뒤덮인 골목이다. 저렴한 가격에 다양한 종류의 먹거리를 만날 수 있다는 것이 서면 먹자골목의 특징이다. 기회를 놓치지 말고 조금씩 여러 음식을 맛보고 주변 사람들과 맛에 대한 진지하고도 재미있는 순위를 매겨보자.
메뉴: 
영업시간: 
주소: 부산시 부산진구 부전동
연락처: 

장소명: 송상현광장
구분: 관광지
카테고리: 
설명: 도심 한가운데 아름다운 숲과 실개천을 간직한 송상현광장, 산들바람 부는 날에 메타세쿼이아 나무 아래에서 나만의 여유를 즐기기에 안성맞춤인 곳이다. 카페와 야외공연장 등 편의시설도 갖추고 있어서 다양한 이벤트와 공연을 관람할 수 있다.
메뉴: 
영업시간: 
주소: 부산시 부산진구 동성로112번길 121-1
연락처: 

장소명: 부전마켓타운
구분: 관광지
카테고리: 
설명: 한국전쟁 이후 피난민들이 모여 일군 삶의 터전, 부전시장. 지금은 부전마켓타운이라는 다소 현대적인 이름을 가졌지만 부산 사람들은 여전히 부전시장이라 부른다. 이른 새벽부터 농수산물 직거래로 시장은 활기가 넘친다. 각양각색 장터음식은 물론 각종 생활용품까지 시장 곳곳을 구경하는 것만으로도 활력충전이 되는 곳이다. 부전마켓타운에는 최고급 보양식품인 인삼이 신선하게 유통된다. 향을 맡는 것만으로도 에너지가 충전되는 기분을 느껴보자.
메뉴: 
영업시간: 
주소: 부산시 부산진구 서전로58번길 51
연락처: 

장소명: 전포사잇길
구분: 관광지
카테고리: 
설명: 전포공구길과 비슷한 분위기 같아 보이지만 더 개성 강한 공간이 존재한다. 전포동을 더욱 동적으로 만들어 주는 개성 강한 가게들은 인더스트리얼 인테리어 분위기가 물씬 풍기는 짙은 레트로 감성이 특징이다.
메뉴: 
영업시간: 
주소: 부산시 부산진

In [41]:
full_text = refined_pre_text + "\n\n" + refined_mid_text + "\n\n" + refined_post_text

In [42]:
full_text

"장소명: 전포공구길\n구분: 관광지\n카테고리: \n설명: 생겨나면서 가장 인기 있는 문화 골목이 되었다. 동시에 아직까지 남아있는 옛 공구 상가의 모습은 전포동의 역사를 그대로 엿볼 수 있는데 이런 것들이 자연스레 어우러져 색다른 골목 분위기를 만들고 있다. 알록달록한 색깔의 외관, 아기자기한 소품, 예쁜 조명, 화려한 장식 등 시선을 사로잡는 다양한 가게들이 속속 들어선 전포공구길만의 공간은 인증사진을 남기고 싶어 하는 여행자들에게 많은 사랑을 받고 있다. '깡깡'하며 기계 고치는 소리와 '드르르륵' 커피 내리는 소리가 과거와 현재를 이어주는 묘한 연결고리로 다가온다. 과거 전포공구길은 차량 재생창, 공업사 등의 시설에서 나온 부품들이 거래되면서 공구 골목을 형성하였다. 전포동 공구 산업이 활발하던 시절 공업사 종사자는 바쁜 일상으로 인해 빵과 우유로 끼니를 해결한 추억이 있다. 최근에는 곳곳에 카페, 맛집, 편집샵들이 생겨나 이색적이고 독특한 분위기로 주목을 받으며 젊은 사람들이 모여들어 힙한 골목길로 자리잡고 있다. 트렌디한 카페와 공방이 하나둘씩 들어서면서 ‘힙’한 거리로 바뀌고 있는 전포공구길은 국내외 관광객이 찾는 핫플레이스가 되었다. 각자의 개성을 뽐내는 카페와 베이커리, 아기자기한 물건들로 가득한 소품샵, 발란사나 아더에러 같은 핫한 브랜드 상점, 다양한 나라의 음식점, 팝업 스토어 등이 전포공구길의 탄생\n메뉴: \n영업시간: \n주소:\n연락처:\n\n장소명: 경성72스튜디오\n구분: 체험\n카테고리: \n설명: fun play Shutter, 놀고 기록하고 사랑하라\n메뉴: \n영업시간: \n주소: 서전로37번길 14 지하층\n연락처: \n\n장소명: 수에노센트스튜디오\n구분: 체험\n카테고리: \n설명: 전문 조향사의 캔들공방\n메뉴: \n영업시간: \n주소: 서전로37번길 26 A동 159호\n연락처: \n\n장소명: 태한사진관\n구분: 체험\n카테고리: \n설명: 소중한 순간을 기록하는 가족사진관\n메뉴: \n영업시간: \n주소: 

## DF로 변환 후 결과 확인 
- GT를 활용한 검증(선택)

In [43]:
pdf_reader.change_str_to_df(full_text).head()

,장소명,구분,카테고리,설명,메뉴,영업시간,주소,연락처
0,전포공구길,관광지,,생겨나면서 가장 인기 있는 문화 골목이 되었다. 동시에 아직까지 남아있는 옛 공구 ...,,,,
1,경성72스튜디오,체험,,"fun play Shutter, 놀고 기록하고 사랑하라",,,서전로37번길 14 지하층,
2,수에노센트스튜디오,체험,,전문 조향사의 캔들공방,,,서전로37번길 26 A동 159호,
3,태한사진관,체험,,소중한 순간을 기록하는 가족사진관,,,서전로47번길 27,
4,어피얼,체험,,오늘을 기억할 수 있는 사진,,,전포대로255번길 33 B동 206호,


In [100]:
# pdf_reader.evaluate_df()

In [106]:
pdf_reader.df.loc[78:, "구분"] = "관광지"

In [108]:
pdf_reader.df.head()

,장소명,구분,카테고리,설명,메뉴,영업시간,주소,연락처
0,금정산성,관광지,,국내에서 가장 큰 규모의 산성이다. 삼국시대에 축조되었으나 일제 암흑기에 일본인들에...,,,부산시 금정구 북문로 78-5 일대,
1,동래읍성,관광지,,부산광역시 기념물 제5호로 지정된 읍성. 임진왜란 때 동래성 전투가 있었던 곳으로 ...,,,부산시 동래구 명륜동 산48-2 일대,051-550-6634
2,영도대교,관광지,,47년 만에 도개가 회복된 부산 최초의 연륙교. 한국전쟁 당시 영도다리 난간에 이름...,매주 토요일 오후 2시부터 약 15분간 도개가 진행된다.,,부산시 영도구 태종로(대교동1가),
3,초량 이바구길,관광지,,부산의 역사와 문화를 담고 있는 테마 거리이다. 일제강점기 부산항 개항을 시작으로 ...,,,부산시 동구 초량상로 49 일대,
4,감천문화마을,관광지,,1950년대 6.25 피난민의 삶의 터전에서 시작된 감천문화 마을은 원래 부산의 모...,,,부산시 사하구 감내2로 203 감천문화마을안내센터,051-204-1444


In [110]:
pdf_reader.df[pdf_reader.df["장소명"] == "영도대교"]

pdf_reader.df.loc[2, "설명"] = pdf_reader.df.loc[2, "설명"] + pdf_reader.df.loc[2, "메뉴"]
pdf_reader.df.loc[2, "메뉴"] = ""



In [111]:
pdf_reader.df[pdf_reader.df["장소명"] == "영도대교"]


,장소명,구분,카테고리,설명,메뉴,영업시간,주소,연락처
2,영도대교,관광지,,47년 만에 도개가 회복된 부산 최초의 연륙교. 한국전쟁 당시 영도다리 난간에 이름...,,,부산시 영도구 태종로(대교동1가),


In [115]:
pdf_reader.df[pdf_reader.df["장소명"] == "금강 식물원"]


,장소명,구분,카테고리,설명,메뉴,영업시간,주소,연락처,출처
39,금강 식물원,관광지,,,,"운영 시간 : 08:00~17:00 / 대인 1,000원 / 청소년 600원 / 어...",부산시 금정구 장전동 산45-49,051-582-3284,Theme(2025)


In [32]:
# 데이터프레임의 각 행을 순회하며 상세 정보 출력
for _, row in pdf_reader.df.iterrows():
    print(f"{row['장소명']}")
    print(f"구분: {row['구분']}")
    print(f"카테고리: {row['카테고리']}")
    print(f"설명: {row['설명']}")
    print(f"메뉴: {row['메뉴']}")
    print(f"영업시간: {row['영업시간']}")
    print(f"주소: {row['주소']}")
    print(f"연락처: {row['연락처']}")
    print("-" * 50)  # 구분을 위한 긴 점선

영도
구분: 관광지
카테고리: 
설명: 때 묻지 않은 순수함을 간직한 영도 봉산마을은 최근 핫한 공간들이 마구 생겨나고 있다. 부산 영도 봉래산 아래 언덕, 심상치 않은 집들이 빼곡하게 모여 있는 봉산마을. 과거, 호황을 이루던 조선업 현장 근로자들이 모여 살던 이곳은 조선업의 불황과 뉴타운 해제 지역으로 선정되면서 한때 많은 사람이 떠났던 곳이다. 하지만 숨겨진 매력을 숨길 수는 없는 법! 텅 빈 집은 주민과 관광객이 찾기 좋은 카페와 서점 등으로 채워졌고 적막했던 거리는 사람들의 웃음소리로 가득해졌다. 미로같이 좁지만 골목 사이 보이는 청아한 영도 앞바다, 골목골목마다 머무르는 햇살 등 곳곳에 숨어있는 매력을 하나씩 찾아보는 묘미가 있다. 다채로운 집들이 만들어내는 스카이라인과 부산항의 풍경이 꽤 매력적인 곳에 입소문을 듣고 찾아온 청년들이 하나 둘 모여 손이 즐거운 봉산마을의 탄생 공방, 입이 즐거운 체험 공간들이 생겨났다. 텅 빈 공간에 숨결을 더해 문화예술로 꽃피우며 아름다운 마을이 변해가는 모습을 보는 것은 언제나 즐겁다. 가족, 친구, 연인, 나홀로 여행을 하기 너무 좋은 숙소들이 많아 한 번 오면 반드시 좋은 기억으로 다시 방문하게 되는 봉산마을은 부산여행의 필수코스가 되었다. 영도에 들어서기 전 인근에 있는 관광지들도 놓치지 말 것. 갓 잡아 온 해산물이 가득한 자갈치시장, 눈물로 가득했던 역사의 영도대교, 부산의 역사가 녹아있는 국제시장까지!
메뉴: 
영업시간: 
주소: 
연락처: 
--------------------------------------------------
자갈치시장
구분: 관광지
카테고리: 
설명: 갓 잡아 온 해산물이 가득한 자갈치시장
메뉴: 
영업시간: 
주소: 
연락처: 
--------------------------------------------------
영도대교
구분: 관광지
카테고리: 
설명: 눈물로 가득했던 역사의 영도대교
메뉴: 
영업시간: 
주소: 
연락처: 
------------------------

## 출처 칼럼 추가

In [44]:
pdf_reader.df["출처"] = "부산한골목-전포공구길"
pdf_reader.df.head()

,장소명,구분,카테고리,설명,메뉴,영업시간,주소,연락처,출처
0,전포공구길,관광지,,생겨나면서 가장 인기 있는 문화 골목이 되었다. 동시에 아직까지 남아있는 옛 공구 ...,,,,,부산한골목-전포공구길
1,경성72스튜디오,체험,,"fun play Shutter, 놀고 기록하고 사랑하라",,,서전로37번길 14 지하층,,부산한골목-전포공구길
2,수에노센트스튜디오,체험,,전문 조향사의 캔들공방,,,서전로37번길 26 A동 159호,,부산한골목-전포공구길
3,태한사진관,체험,,소중한 순간을 기록하는 가족사진관,,,서전로47번길 27,,부산한골목-전포공구길
4,어피얼,체험,,오늘을 기억할 수 있는 사진,,,전포대로255번길 33 B동 206호,,부산한골목-전포공구길


## 설명이 없는 항목 삭제

In [21]:
is_empty_desc = pdf_reader.df['설명'].isna() | (pdf_reader.df['설명'].str.strip() == "")
deleted_df = pdf_reader.df[is_empty_desc]

deleted_df

,장소명,구분,카테고리,설명,메뉴,영업시간,주소,연락처,출처
13,망미골목,관광지,,,,,,,부산한골목-망미골목


In [22]:
len(pdf_reader.df)

34

In [23]:
pdf_reader.df = pdf_reader.df[~is_empty_desc].reset_index(drop=True)

len(pdf_reader.df)

33

## json 파일 생성

In [45]:
file_name = "./busan_trip_jsons/busan_jeonpo.json"

pdf_reader.save_to_json(file_name)

✅ JSON 저장 완료: ./busan_trip_jsons/busan_jeonpo.json
